<a href="https://colab.research.google.com/github/Nakib-Nasrullah/Heart_disease/blob/main/final_defense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas==2.2.2 wfdb==3.4.1 scipy matplotlib

import wfdb
import numpy as np
import pandas as pd
import os

In [2]:
DATA_DIR = "mitdb"

if not os.path.exists(DATA_DIR):
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR)

print("Dataset ready!")

Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

In [3]:
WINDOW = 187
HALF = WINDOW // 2

label_map = {
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
    'A': 1, 'a': 1, 'J': 1, 'S': 1,
    'V': 2, 'E': 2,
    'F': 3
}

beats = []

records = sorted([f.split('.')[0] for f in os.listdir(DATA_DIR) if f.endswith('.dat')])

for record in records:
    try:
        signal, _ = wfdb.rdsamp(os.path.join(DATA_DIR, record))
        ann = wfdb.rdann(os.path.join(DATA_DIR, record), 'atr')
    except:
        continue

    ecg = signal[:, 0]

    for r, sym in zip(ann.sample, ann.symbol):
        if sym not in label_map:
            continue

        if r - HALF < 0 or r + HALF >= len(ecg):
            continue

        beat = ecg[r-HALF:r+HALF+1]
        beats.append([record] + beat.tolist() + [label_map[sym]])

columns = ["record_id"] + [f"f{i}" for i in range(WINDOW)] + ["label"]
df = pd.DataFrame(beats, columns=columns)

df.to_csv("mitbih_patient_level.csv", index=False)

print("Dataset created:", df.shape)

Dataset created: (101426, 189)


In [4]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("mitbih_patient_level.csv")

patients = df['record_id'].unique()

# 80% train+val, 20% test
trainval_patients, test_patients = train_test_split(
    patients,
    test_size=0.20,
    random_state=42
)

# 10% of 80% → validation
train_patients, val_patients = train_test_split(
    trainval_patients,
    test_size=0.10,
    random_state=42
)

train_df = df[df['record_id'].isin(train_patients)]
val_df   = df[df['record_id'].isin(val_patients)]
test_df  = df[df['record_id'].isin(test_patients)]

# Check
assert set(train_df['record_id']).isdisjoint(val_df['record_id'])
assert set(train_df['record_id']).isdisjoint(test_df['record_id'])
assert set(val_df['record_id']).isdisjoint(test_df['record_id'])

print("No patient overlap ✔")

# Save
train_df.to_csv("mitbih_train.csv", index=False)
val_df.to_csv("mitbih_val.csv", index=False)
test_df.to_csv("mitbih_test.csv", index=False)

print("Split done!")

No patient overlap ✔
Split done!


In [5]:
train_df = pd.read_csv("mitbih_train.csv")
val_df   = pd.read_csv("mitbih_val.csv")
test_df  = pd.read_csv("mitbih_test.csv")

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (72737, 189)
Val: (8245, 189)
Test: (20444, 189)


In [6]:
X_train = train_df.iloc[:, 1:-1].values
y_train = train_df['label'].values.astype(int)

X_val = val_df.iloc[:, 1:-1].values
y_val = val_df['label'].values.astype(int)

X_test = test_df.iloc[:, 1:-1].values
y_test = test_df['label'].values.astype(int)

In [7]:
X_train = (X_train - X_train.mean(axis=1, keepdims=True)) / \
          (X_train.std(axis=1, keepdims=True) + 1e-8)

X_val = (X_val - X_val.mean(axis=1, keepdims=True)) / \
        (X_val.std(axis=1, keepdims=True) + 1e-8)

X_test = (X_test - X_test.mean(axis=1, keepdims=True)) / \
         (X_test.std(axis=1, keepdims=True) + 1e-8)

In [8]:
X_train = X_train.reshape(-1, 187, 1)
X_val   = X_val.reshape(-1, 187, 1)
X_test  = X_test.reshape(-1, 187, 1)

print(X_train.shape, X_val.shape, X_test.shape)

(72737, 187, 1) (8245, 187, 1) (20444, 187, 1)


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Conv1D(32, 5, activation='relu', input_shape=(187, 1)),
    BatchNormalization(),
    MaxPooling1D(2),

    Conv1D(64, 5, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),

    Conv1D(128, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(4, activation='softmax')
])

model.compile(
    optimizer=Adam(0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 183, 32)        │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 183, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 91, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 87, 64)         │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 87, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 43, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 41, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 41, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2560)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       327,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 364,420 (1.39 MB)

 Trainable params: 363,972 (1.39 MB)

 Non-trainable params: 448 (1.75 KB)

In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),  # ✅ CORRECT
    verbose=1
)

Epoch 1/20
2274/2274 ━━━━━━━━━━━━━━━━━━━━ 92s 38ms/step - accuracy: 0.9733 - loss: 0.1117 - val_accuracy: 0.7606 - val_loss: 3.1125
Epoch 2/20
2274/2274 ━━━━━━━━━━━━━━━━━━━━ 82s 36ms/step - accuracy: 0.9841 - loss: 0.0629 - val_accuracy: 0.7511 - val_loss: 2.4607
Epoch 3/20
2274/2274 ━━━━━━━━━━━━━━━━━━━━ 143s 37ms/step - accuracy: 0.9864 - loss: 0.0533 - val_accuracy: 0.7537 - val_loss: 2.9561
Epoch 4/20
2274/2274 ━━━━━━━━━━━━━━━━━━━━ 82s 36ms/step - accuracy: 0.9891 - loss: 0.0431 - val_accuracy: 0.7551 - val_loss: 2.7827
Epoch 5/20
2274/2274 ━━━━━━━━━━━━━━━━━━━━ 81s 36ms/step - accuracy: 0.9895 - loss: 0.0372 - val_accuracy: 0.7527 - val_loss: 2.4799
Epoch 6/20
2274/2274 ━━━━━━━━━━━━━━━━━━━━ 81s 36ms/step - accuracy: 0.9911 - loss: 0.0323 - val_accuracy: 0.7499 - val_loss: 3.3065
Epoch 7/20
2274/2274 ━━━━━━━━━━━━━━━━━━━━ 94s 41ms/step - accuracy: 0.9914 - loss: 0.0321 - val_accuracy: 0.7560 - val_loss: 4.2920
Epoch 8/20
2274/2274 ━━━━━━━━━━━━━━━━━━━━ 131s 36ms/step - accuracy: 0.9922

In [11]:
model.save("mitbih_cnn_patient_independent.keras")
print("Model saved")

Model saved


In [12]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print("Patient-Independent Test Accuracy:", test_acc)

Patient-Independent Test Accuracy: 0.8231754899024963
